# P0 · The 45-minute skeleton (memorize this order)

**ID:** P0 · **Target:** 45 min · **Plan week:** 1 (Saturday: write it from memory), 6, 9 and 16 (full mock runs on unseen datasets) · **Part:** P (PyTorch first principles)

**The drill.** This is the "blank Colab" interview in its full form. Build a 10-class classifier on
scikit-learn's bundled `load_digits` (1797 8×8 grey images, pixel values 0-16, offline) by writing the
**eight steps of guide §P0 in order**, one cell per step, talking out loud the whole time:

1. Data: load, inspect shapes/dtypes/class balance, stratified train/val/test split, normalize with train statistics only.
2. `Dataset` / `DataLoader` (shuffle train only, sensible batch size, `collate_fn` if lengths vary).
3. Model as an `nn.Module`; output raw logits; count parameters.
4. Loss chosen from the task (P1 table), optimizer (AdamW default) and LR.
5. Training loop: `model.train()`, to device, forward, loss, `zero_grad`, `backward`, (clip), `step`; log the loss.
6. Eval loop: `model.eval()` + `torch.no_grad()`, metrics aggregated over the whole set.
7. Sanity checks: initial loss vs ln C, overfit one batch, read the curves for over/underfitting.
8. Improve and discuss: regularization, LR schedule, early stopping, checkpointing, error analysis.

The sanity checks (step 7) run **before** the long training run: they are cheap and catch most bugs.
After the skeleton there is a short **loss-selection quiz** (guide §P1) and a list of **unseen datasets for the
week-6 mock runs**.

**Rules.** No docs, no LLM. Each solution cell keeps only its spec (comments). The self-check cell names
the variables it expects. Then answer the questions out loud and log the session.

## Sources
- PyTorch tutorial, *Learn the Basics* (Datasets & DataLoaders, Build the Model, Optimization loop):
  https://docs.pytorch.org/tutorials/beginner/basics/intro.html (BSD-3; skeleton order follows its train/test loops).
- PyTorch tutorial, *Saving and Loading Models* (state_dict, general checkpoint with optimizer):
  https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
- A. Karpathy, *A Recipe for Training Neural Networks* (2019): "verify loss @ init", "overfit one batch",
  "visualize prediction dynamics": https://karpathy.github.io/2019/04/25/recipe/
- V. Godbole, G. Dahl, J. Gilmer, C. Shallue, Z. Nado, *Deep Learning Tuning Playbook* (2023),
  "Saving checkpoints and retrospectively selecting the best checkpoint":
  https://github.com/google-research/tuning_playbook (commit 7554a51, `README.md`, CC-BY-4.0).
- I. Loshchilov, F. Hutter, *Decoupled Weight Decay Regularization* (AdamW), ICLR 2019, arXiv:1711.05101 —
  https://arxiv.org/abs/1711.05101
- I. Loshchilov, F. Hutter, *SGDR: Stochastic Gradient Descent with Warm Restarts* (cosine schedule), ICLR 2017,
  arXiv:1608.03983 — https://arxiv.org/abs/1608.03983
- scikit-learn `load_digits` (copy of the UCI "Optical Recognition of Handwritten Digits" test set):
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html ·
  https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits
- `torch.nn.CrossEntropyLoss` docs (class-index `long` targets, `label_smoothing`, `ignore_index`):
  https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html

In [ ]:
import copy
import math
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from prep import Session, check, check_close, seed_everything

s = Session("P0", target_min=45)
seed_everything(0)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. DATA. X, y = sklearn.datasets.load_digits(return_X_y=True). Print shapes, dtypes, value range, class counts.
#    Stratified split 70/15/15 with train_test_split(..., stratify=..., random_state=0) twice
#    -> X_train, X_val, X_test, y_train, y_val, y_test (numpy).
#    Standardize per feature with TRAIN mean/std only -> `mu`, `sd`. Some pixels are always 0 in train:
#    guard the std (e.g. sd[sd < 1e-6] = 1) so nothing becomes NaN.
#    Convert X to float32 tensors, y to int64 (long) tensors.
# YOUR CODE HERE


In [ ]:
# 2. DATASET / DATALOADER. Write a map-style `Dataset` subclass `ArrayDataset(X, y)` with __len__ and
#    __getitem__ returning (x_i, y_i). Build train_ds/val_ds/test_ds and DataLoaders:
#    train_dl (batch 64, shuffle=True), val_dl and test_dl (batch 512, shuffle=False).
#    (No collate_fn needed: every example has the same shape. Say when you WOULD need one.)
# YOUR CODE HERE


In [ ]:
# 3. MODEL. `class MLP(nn.Module)`: 64 -> 128 -> 128 -> 10, ReLU, Dropout(p=0.1) after each hidden layer.
#    forward returns RAW LOGITS of shape (B, 10) (no softmax). `make_model()` returns a fresh one on `dev`.
#    `model = make_model()`; print the parameter count.
# YOUR CODE HERE


In [ ]:
# 4. LOSS + OPTIMIZER. `loss_fn = nn.CrossEntropyLoss()` (targets: long class indices, shape (B,)).
#    `opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)`.
# YOUR CODE HERE


In [ ]:
# 5. TRAIN LOOP. `train_one_epoch(model, dl, opt, clip=1.0) -> mean train loss` (weighted by batch size).
#    model.train(); per batch: to(dev), forward, loss, opt.zero_grad(), loss.backward(),
#    torch.nn.utils.clip_grad_norm_(model.parameters(), clip), opt.step().
# YOUR CODE HERE


In [ ]:
# 6. EVAL LOOP. `evaluate(model, dl) -> (mean loss, accuracy)` over the WHOLE set:
#    model.eval() + torch.no_grad(); sum loss * batch size and correct counts, divide by N at the end.
#    Also `predict(model, dl) -> (all_logits, all_targets)` concatenated on CPU, for error analysis.
# YOUR CODE HERE


In [ ]:
# 7. SANITY CHECKS (before the long run), then a plain baseline run.
#    (a) `init_loss`, _ = evaluate(model, val_dl); compare with ln 10.
#    (b) Overfit ONE batch of 32 train examples with a fresh model, AdamW lr=1e-2, 300 steps
#        (no dropout effect matters: call .eval() OR keep train mode, your choice; say why). Store `overfit_loss`.
#    (c) Baseline: train `model` for 40 epochs with constant LR; record `history` dict with
#        "train_loss", "val_loss", "val_acc" per epoch. Plot train vs val loss and say what you see.
# YOUR CODE HERE


In [ ]:
# 8. IMPROVE + DISCUSS. Fresh model `best_model` trained with:
#    - AdamW lr=1e-3, wd=5e-2, CosineAnnealingLR over max 60 epochs (scheduler.step() once per EPOCH);
#      record the LR each epoch in a list `lrs`;
#    - early stopping on val LOSS with patience 20: keep a deep copy of the best state_dict,
#      store `best_val_loss`, `best_epoch`, `stopped_epoch`;
#    - checkpoint the best {model, optimizer, epoch} with torch.save to `ckpt_path` inside tempfile.mkdtemp();
#      after training, load it back into a FRESH model -> `best_model`.
#    Then touch the test set ONCE: `test_loss, test_acc = evaluate(best_model, test_dl)`.
#    Error analysis: confusion pairs on test (`pairs`: the 3 most frequent (true, pred) mistakes) and a plot of
#    a few misclassified test images (un-standardize with mu/sd, reshape to 8x8).
# YOUR CODE HERE


## Self-check
Expects: `mu`, `sd`, `X_train`, `y_train`, `train_ds`, `model`, `make_model`, `init_loss`, `overfit_loss`,
`history`, `lrs`, `best_model`, `best_val_loss`, `ckpt_path`, `stopped_epoch`, `test_acc`, `evaluate`, `val_dl`.

In [ ]:
check("no NaN after standardization", not torch.isnan(X_train).any().item(), "guard the zero-std pixels")
check_close("train mean ~ 0 (train stats used)", X_train.mean(0), torch.zeros(X_train.shape[1]), atol=1e-4)
check("targets are long class indices", y_train.dtype == torch.long and y_train.ndim == 1, f"got {y_train.dtype}, shape {tuple(y_train.shape)}")
check("Dataset length/item", len(train_ds) == len(X_train) and len(train_ds[0]) == 2)
check("model outputs (B, 10) logits", model(X_train[:7].to(dev)).shape == (7, 10))
check("no Softmax layer in the model", not any(isinstance(m, (nn.Softmax, nn.LogSoftmax)) for m in model.modules()))
check("initial loss ~ ln 10", abs(init_loss - math.log(10)) < 0.3, f"got {init_loss:.3f}")
check("can overfit one batch", overfit_loss < 0.05, f"got {overfit_loss:.4f}")
check("history has one entry per epoch", len(history["train_loss"]) == len(history["val_loss"]) == len(history["val_acc"]) > 0)
check("LR schedule actually decays", len(lrs) > 1 and lrs[-1] < lrs[0], f"lrs[0]={lrs[0]:.2e}, lrs[-1]={lrs[-1]:.2e}")
reloaded = make_model()
reloaded.load_state_dict(torch.load(ckpt_path, map_location=dev)["model"])
check_close("checkpoint reproduces best val loss", evaluate(reloaded, val_dl)[0], best_val_loss, atol=1e-5)
check("checkpoint also stores the optimizer state", "opt" in torch.load(ckpt_path, map_location=dev) or "optimizer" in torch.load(ckpt_path, map_location=dev))
check("eval is deterministic (dropout off in eval)", evaluate(best_model, val_dl) == evaluate(best_model, val_dl))
check("test accuracy > 0.93", test_acc > 0.93, f"got {test_acc:.3f}")

## Loss-selection quiz (guide §P1)
For each task say, in one breath: **model output shape, loss (PyTorch class), target dtype + shape**, and the trap.
1. Spam vs not spam, one score per email.
2. ImageNet classification, 1000 classes, one label per image.
3. Tag a photo with any subset of 20 attributes.
4. Next-token prediction for a GPT, batch B, length T, vocab V, right-padded sequences.
5. Semantic segmentation into C classes on H×W images.
6. Predict house price; also predict an uncertainty for it.
7. Predict number of clicks per hour.
8. Distil a teacher's softmax distribution into a student.
9. Train image/text embeddings so matching pairs are close.
10. 0.1% fraud transactions; the model keeps predicting "not fraud".

## Unseen datasets for mock runs (weeks 6, 10, 16)
Pick one you have not used for a while, start a 45-minute timer, and run the eight steps from a blank notebook.
All load offline (sklearn bundles the small ones; the rest are in `prep.DATA`).

| Dataset | Task | One-line loader |
| --- | --- | --- |
| Breast cancer (569×30) | binary, mild imbalance | `X, y = sklearn.datasets.load_breast_cancer(return_X_y=True)` |
| Wine (178×13) | 3-class, tiny: use k-fold | `X, y = sklearn.datasets.load_wine(return_X_y=True)` |
| Digits as regression | regress the digit value 0-9 (MSE vs Huber) | `X, y = sklearn.datasets.load_digits(return_X_y=True); y = y.astype("float32")` |
| Diabetes (442×10) | regression, small and noisy | `X, y = sklearn.datasets.load_diabetes(return_X_y=True)` |
| California housing (20640×8) | regression, capped target | `sklearn.datasets.fetch_california_housing(data_home=DATA/"sklearn", download_if_missing=False)` |
| FashionMNIST | 10-class images | `torchvision.datasets.FashionMNIST(DATA, train=True, download=False)` |
| MNIST | 10-class images (or a VAE) | `torchvision.datasets.MNIST(DATA, train=True, download=False)` |
| AG News subset | 4-class text | `pd.read_parquet(DATA/"ag_news"/"train.parquet").sample(20000, random_state=0)` |
| Synthetic multilabel | 5-10 labels per row | `sklearn.datasets.make_multilabel_classification(n_samples=5000, n_classes=8, random_state=0)` |
| Synthetic 1 % positives | imbalanced binary | `sklearn.datasets.make_classification(n_samples=50000, weights=[0.99], random_state=0)` |

## Say it out loud (P3)
1. Walk through the eight steps in order without notes. Which one would you skip under time pressure, and which never?
2. Why is the initial loss ln 10 here, and what would an initial loss of 5 tell you?
3. Why did the overfit-one-batch check turn dropout off? What does it mean if you cannot overfit 32 examples?
4. What do `model.train()` / `model.eval()` change in *this* model, and what does `torch.no_grad()` save?
5. AdamW vs Adam with L2 regularization: what is the difference? Why AdamW as the default, and what LR would you start at?
6. Read the baseline curves: is it over- or underfitting? What would you try first, and why?
7. Early stopping with a cosine schedule: what is subtle about stopping early when the schedule assumes a fixed length?
8. What goes in a checkpoint if you want to *resume* training, not just evaluate?
9. When do you need a `collate_fn`?

In [ ]:
s.finish(lookups=0, score=None, notes="")  # score: 1-5 self-rating; notes: what tripped you up